In [1]:
print('Hi')

Hi


In [2]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.13.5 environment at: D:\AMARNATH_NEW\LLMOPS\.venv
Resolved 66 packages in 847ms
 Downloaded shapely
 Downloaded sympy
 Downloaded pillow
 Downloaded onnxruntime
 Downloaded rapidocr-onnxruntime
 Downloaded opencv-python
Prepared 9 packages in 28.16s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 9 packages in 14.48s
 + flatbuffers==25.12.19
 + mpmath==1.3.0
 + onnxruntime==1.24.4
 + opencv-python==4.13.0.92
 + pillow==12.2.0
 + pyclipper==1.4.0
 + rapidocr-onnxruntime==1.4.4
 + shapely==2.1.2
 + sympy==1.14.0


In [4]:
#setup env variables
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

In [5]:
#Data Ingestion - Ingesting the data into the vector store
from langchain.document_loaders import TextLoader

In [6]:
loader = TextLoader("D:\AMARNATH_NEW\LLMOPS\data\AI.txt",encoding = "utf-8")
documents = loader.load()

<>:1: SyntaxWarning: invalid escape sequence '\A'
<>:1: SyntaxWarning: invalid escape sequence '\A'
C:\Users\amarn\AppData\Local\Temp\ipykernel_11068\1649414557.py:1: SyntaxWarning: invalid escape sequence '\A'
  loader = TextLoader("D:\AMARNATH_NEW\LLMOPS\data\AI.txt",encoding = "utf-8")


In [7]:
documents

[Document(metadata={'source': 'D:\\AMARNATH_NEW\\LLMOPS\\data\\AI.txt'}, page_content='Artificial Intelligence (AI) is a branch of computer science dedicated to creating systems capable of performing tasks that typically require human intelligence. These tasks include learning, reasoning, problem-solving, perception, and understanding language. Unlike traditional software that follows strict, pre-programmed rules, AI systems analyze vast amounts of data to identify patterns, allowing them to make predictions, adapt to new inputs, and improve over time. \n\nHere is a detailed breakdown of AI:\nHow AI Works\nAI works by combining large datasets with intelligent, iterative algorithms. \n\nData Ingestion: AI systems are fed massive amounts of data (text, images, numbers).\nPattern Recognition: Algorithms scan this data to identify patterns and relationships.\nModel Training: The system uses these patterns to create a model—a set of rules—that can interpret new, unseen data.\nPrediction/Act

In [8]:
documents[0].page_content

'Artificial Intelligence (AI) is a branch of computer science dedicated to creating systems capable of performing tasks that typically require human intelligence. These tasks include learning, reasoning, problem-solving, perception, and understanding language. Unlike traditional software that follows strict, pre-programmed rules, AI systems analyze vast amounts of data to identify patterns, allowing them to make predictions, adapt to new inputs, and improve over time. \n\nHere is a detailed breakdown of AI:\nHow AI Works\nAI works by combining large datasets with intelligent, iterative algorithms. \n\nData Ingestion: AI systems are fed massive amounts of data (text, images, numbers).\nPattern Recognition: Algorithms scan this data to identify patterns and relationships.\nModel Training: The system uses these patterns to create a model—a set of rules—that can interpret new, unseen data.\nPrediction/Action: Once trained, the AI can make decisions, create content, or solve problems based 

In [15]:
#chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 70,
    chunk_overlap = 20,
    #length_function = len
)

In [16]:
text_chunks = text_splitter.split_documents(documents)

In [17]:
text_chunks

[Document(metadata={'source': 'D:\\AMARNATH_NEW\\LLMOPS\\data\\AI.txt'}, page_content='Artificial Intelligence (AI) is a branch of computer science dedicated'),
 Document(metadata={'source': 'D:\\AMARNATH_NEW\\LLMOPS\\data\\AI.txt'}, page_content='science dedicated to creating systems capable of performing tasks'),
 Document(metadata={'source': 'D:\\AMARNATH_NEW\\LLMOPS\\data\\AI.txt'}, page_content='of performing tasks that typically require human intelligence. These'),
 Document(metadata={'source': 'D:\\AMARNATH_NEW\\LLMOPS\\data\\AI.txt'}, page_content='intelligence. These tasks include learning, reasoning,'),
 Document(metadata={'source': 'D:\\AMARNATH_NEW\\LLMOPS\\data\\AI.txt'}, page_content='reasoning, problem-solving, perception, and understanding language.'),
 Document(metadata={'source': 'D:\\AMARNATH_NEW\\LLMOPS\\data\\AI.txt'}, page_content='language. Unlike traditional software that follows strict,'),
 Document(metadata={'source': 'D:\\AMARNATH_NEW\\LLMOPS\\data\\AI.txt'},

In [18]:
! uv pip install faiss-cpu

Using Python 3.13.5 environment at: D:\AMARNATH_NEW\LLMOPS\.venv
Checked 1 package in 75ms


In [20]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

#embeddings = OpenAIEmbeddings()

In [ ]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [ ]:
vectorstore

In [ ]:
retriever=vectorstore.as_retriever()

In [ ]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)


In [ ]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [ ]:
prompt=ChatPromptTemplate.from_template(template)

In [ ]:
prompt

In [ ]:
from langchain.schema.output_parser import StrOutputParser

In [ ]:
output_parser=StrOutputParser()

In [ ]:
from langchain.chat_models import ChatOpenAI
llm_model=ChatOpenAI(model_name="gpt-4o-mini")

In [ ]:
from langchain.schema.runnable import RunnablePassthrough


In [ ]:
rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [ ]:
rag_chain.invoke("tell me about Agentic AI")